### Model Preprocessing and Spliting Data

Throughout preprocessing, we focused on applying the techniques we learned during the summer course. This included steps such as one-hot encoding, removing outliers using winsorizing, and dropping irrelevant columns. Initially, we overlooked removing several key columns, which led to significant data leakage—specifically from the pledged and backers columns, both of which strongly indicate whether a project is successful. After removing these columns and completing the remaining preprocessing steps, we observed a substantial improvement in our model’s results.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split

In [26]:
df = pd.read_csv('data/DSI_kickstarterscrape_dataset.csv', encoding='ISO-8859-1') # Loading in the dataset
df.head() # seeing the dataset

,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,138119,STRUM,http://www.kickstarter.com/projects/185476022/...,Film & Video,Animation,"Los Angeles, CA",live,20000.0,56.0,0.002800,3,"Fri, 08 Jun 2012 00:00:31 -0000",10,"$1,$10,$25,$40,$50,$100,$250,$1,000,$1,337,$9,001",1,0,28.00
3,237090,GETTING OVER - One son's search to finally kno...,http://www.kickstarter.com/projects/charnick/g...,Film & Video,Documentary,"Los Angeles, CA",successful,6000.0,6535.0,1.089167,100,"Sun, 08 Apr 2012 02:14:00 -0000",13,"$1,$10,$25,$30,$50,$75,$85,$100,$110,$250,$500...",4,0,32.22
4,246101,The Launch of FlyeGrlRoyalty &quot;The New Nam...,http://www.kickstarter.com/projects/flyegrlroy...,Fashion,Fashion,"Novi, MI",failed,3500.0,0.0,0.000000,0,"Wed, 01 Jun 2011 15:25:39 -0000",6,"$10,$25,$50,$100,$150,$250",2,0,30.00


In [27]:
df.drop(columns = ['name', 'url', 'project id', 'reward levels'], inplace= True) # droping irrelvant columns before dropping rows with missing values

In [28]:
np.sum(df.isnull(), axis = 0) # seeing the number of missing values

category                0
subcategory             0
location             1322
status                  0
goal                    0
pledged                12
funded percentage       0
backers                 0
funded date             0
levels                  0
updates                 0
comments                0
duration                0
dtype: int64

In [29]:
df.dropna(inplace= True) # dropping all rows with missing values

In [30]:
np.sum(df.isnull(), axis = 0) # checking there are no missing values

category             0
subcategory          0
location             0
status               0
goal                 0
pledged              0
funded percentage    0
backers              0
funded date          0
levels               0
updates              0
comments             0
duration             0
dtype: int64

In [31]:
df = df[df['location'].str.match(r".*, [A-Z]{2}$")] # making sure there are only US locations as they end in , XX

In [32]:
df.nunique() # to see which values I should one-hot encode

category                14
subcategory             51
location              4034
status                   5
goal                  1714
pledged              10424
funded percentage    20462
backers                933
funded date          38004
levels                  62
updates                 82
comments               298
duration              5649
dtype: int64

In [33]:
df['state'] = df['location'].str.extract(r", ([A-Z]{2})$") # creating a new state column by extracting the states

In [34]:
df.nunique() # state has a lot less unique values so it is okay to one-hot encode it

category                14
subcategory             51
location              4034
status                   5
goal                  1714
pledged              10424
funded percentage    20462
backers                933
funded date          38004
levels                  62
updates                 82
comments               298
duration              5649
state                   51
dtype: int64

In [35]:
df.drop(columns = ['location'], inplace= True) # only going to use the state

In [36]:
df.shape # seeing the shape before one-hot encoding

(42258, 13)

In [37]:
df = pd.get_dummies(df, columns=['category', 'subcategory', 'state'], drop_first=True, dtype=int) # one-hot encoding all these columns using 1s and 0s

In [38]:
df.shape # seeing the increase in rows 

(42258, 123)

In [39]:
df['funded date'] = pd.to_datetime(df['funded date'], errors='coerce') # turns the rows into date time objects
df['funded_month'] = df['funded date'].dt.month # We can now just extract the month
df.drop(columns = ['funded date'], inplace= True) # drop this column as it is no longer needed

In [40]:
df.head() # just seeing what the dataset looks like at this stage 

,status,goal,pledged,funded percentage,backers,levels,updates,comments,duration,category_Comics,...,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,funded_month
0,successful,10500.0,11545.0,1.099524,66,7,10,2,30.00,0,...,0,0,0,0,0,0,0,0,0,8
1,failed,4000.0,20.0,0.005000,2,5,6,0,47.18,0,...,0,0,0,0,0,0,0,0,0,8
2,live,20000.0,56.0,0.002800,3,10,1,0,28.00,0,...,0,0,0,0,0,0,0,0,0,6
3,successful,6000.0,6535.0,1.089167,100,13,4,0,32.22,0,...,0,0,0,0,0,0,0,0,0,4
4,failed,3500.0,0.0,0.000000,0,6,2,0,30.00,0,...,0,0,0,0,0,0,0,0,0,6


In [41]:
df[['goal', 'pledged', 'backers', 'duration']].describe(percentiles=[0.01, 0.99]) # just looking at the distributions to see if I need to drop any outliers

,goal,pledged,backers,duration
count,4.225800e+04,4.225800e+04,42258.000000,42258.000000
mean,1.206502e+04,4.987742e+03,69.446661,39.615347
std,1.965826e+05,5.903201e+04,710.456596,17.087500
min,1.000000e-02,0.000000e+00,0.000000,1.000000
1%,2.000000e+02,0.000000e+00,0.000000,10.000000
50%,4.000000e+03,1.280000e+03,23.000000,31.530000
99%,1.000000e+05,4.578130e+04,604.430000,90.040000
max,2.147484e+07,1.026684e+07,87142.000000,91.960000


In [42]:
df['goal'] = stats.mstats.winsorize(df['goal'], limits=[0.01, 0.01]) # removing the outliers
df['pledged'] = stats.mstats.winsorize(df['pledged'], limits=[0.01, 0.01])
df['backers'] = stats.mstats.winsorize(df['backers'], limits=[0.01, 0.01])
df['duration'] = stats.mstats.winsorize(df['duration'], limits=[0.01, 0.01])

In [43]:
df[['goal', 'pledged', 'backers', 'duration']].describe(percentiles=[0.01, 0.99]) # seeing the difference

,goal,pledged,backers,duration
count,42258.000000,42258.000000,42258.000000,42258.000000
mean,8734.990622,3760.558143,51.556060,39.637447
std,15034.435287,6932.675505,89.351021,17.005553
min,200.000000,0.000000,0.000000,10.000000
1%,200.000000,0.000000,0.000000,10.000000
50%,4000.000000,1280.000000,23.000000,31.530000
99%,100000.000000,45781.300000,604.430000,90.040000
max,100000.000000,45901.000000,605.000000,90.040000


In [44]:
df["status"].unique() # checking the number of unique values in the label column

array(['successful', 'failed', 'live', 'canceled', 'suspended'],
      dtype=object)

In [45]:
df.shape # checking the number of rows before 

(42258, 123)

In [46]:
df = df[df['status'] != 'live'] 
df = df[df['status'] != 'canceled'] 
df = df[df['status'] != 'suspended']  # deleting the values we do not care about currently

In [47]:
df.shape # checking the number of rows after to see about a 10% loss in number of rows 

(38504, 123)

In [52]:
y = df['status'] # labeling the target
X = df.drop(columns=['status','pledged','funded percentage','backers']) # feature columns
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [53]:
X

,goal,levels,updates,comments,duration,category_Comics,category_Dance,category_Design,category_Fashion,category_Film & Video,...,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,funded_month
0,10500.0,7,10,2,30.00,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,8
1,4000.0,5,6,0,47.18,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,8
3,6000.0,13,4,0,32.22,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,4
4,3500.0,6,2,0,30.00,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,6
5,3500.0,7,8,0,21.43,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45952,500.0,3,2,0,37.83,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,3
45953,10000.0,14,1,1,59.96,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
45954,10000.0,4,2,0,27.32,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
45955,2999.0,7,17,0,30.00,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5
